# 愚公移山 Bucket & Sentiment Analysis

Interactive visualization of sentiment trends, monthly seasonality, and -2 sentiment bucket deep-dive.

In [1]:
# 1. Load Data and Set Up Plotting
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load the results CSV
df = pd.read_csv('results/愚公_buckets_results.csv')
print(f"Loaded {len(df)} posts")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Loaded 4827 posts
Columns: ['post_id', 'time', 'year', 'month', 'text', 'text_length', 'qwen_sentiment', 'qwen_bucket', 'qwen_confidence', 'qwen_error', 'qwen_processed_at']

First few rows:
            post_id                 time    year  month  \
0  3922983548167157  2015-12-22 22:47:00  2015.0   12.0   
1  3926390615591862  2016-01-01 08:25:00  2016.0    1.0   
2  3926416321725250  2016-01-01 10:07:00  2016.0    1.0   
3  3926602234267025  2016-01-01 22:26:00  2016.0    1.0   
4  3926602884890951  2016-01-01 22:29:00  2016.0    1.0   

                                                text  text_length  \
0  #晚安#别人说同病才会学识相爱，若那些阻碍炼制出将来，路障清了，情人怎麽过后来。  K愚公移...           48   
1  新的一年要有新的开始。~啥也看不见就跑跑跑，边喘气儿边唱愚公移山骑车去上早班的阿姨都转过来瞅...           71   
2  #美麗河南#太行山上的絕壁長廊也真是絕了！ 站在崖上、看著行走的車輛，難免心驚膽顫。 山腰中...           89   
3  一个有意义的元旦。‘’齐汇千佛山，祈福旺新年‘’，嘉华人用实际行动践行“做中国最负责任”旅游...          116   
4  父亲的回忆录之《喜鹊搭窝》下：一个人有了期望之后的坚持和耐力能完成让自己都吃惊的事，用愚公移...          119   

   qwen_sentiment qwen_bucket  qwen_con

In [2]:
# 2. Clean Columns and Parse Timestamps
# Ensure sentiment is numeric
df['qwen_sentiment'] = pd.to_numeric(df['qwen_sentiment'], errors='coerce')

# Parse datetime if needed
if 'time' in df.columns:
    df['datetime'] = pd.to_datetime(df['time'], errors='coerce')
else:
    print("WARNING: 'time' column not found")

# Drop rows with invalid sentiment
df_clean = df.dropna(subset=['qwen_sentiment']).copy()
print(f"After removing invalid sentiments: {len(df_clean)} posts")

# Filter to processed rows only
df_clean = df_clean[df_clean['qwen_processed_at'].notna()].copy()
print(f"After filtering processed rows: {len(df_clean)} posts")

print(f"\nSentiment value counts:")
print(df_clean['qwen_sentiment'].value_counts().sort_index())

After removing invalid sentiments: 4827 posts
After filtering processed rows: 4827 posts

Sentiment value counts:
qwen_sentiment
-2       4
-1     323
 0     317
 1    2451
 2    1732
Name: count, dtype: int64


In [3]:
# 3. Build Month-Only Feature (Ignore Year)
# Extract month from datetime (1-12, ignore year)
df_clean['month'] = df_clean['datetime'].dt.month
df_clean['date'] = df_clean['datetime'].dt.date

# Create month name for plotting
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
df_clean['month_name'] = df_clean['month'].map(month_names)

# Sort months in calendar order
month_order = [month_names[i] for i in range(1, 13)]

print("Month distribution (collapsed across years):")
print(df_clean['month_name'].value_counts().reindex(month_order))
print(f"\nDate range: {df_clean['datetime'].min()} to {df_clean['datetime'].max()}")

Month distribution (collapsed across years):
month_name
Jan     505
Feb     311
Mar     256
Apr     396
May     477
Jun     273
Jul     346
Aug     298
Sep     288
Oct     298
Nov     346
Dec    1032
Name: count, dtype: int64

Date range: 2015-12-22 22:47:00 to 2025-04-01 22:56:53


In [8]:
# 4. Visualize Overall Sentiment Trends
# Overall sentiment distribution (bar chart only)
sentiment_counts = df_clean['qwen_sentiment'].value_counts().sort_index()

sentiment_labels = {-2: 'Very Negative (-2)', -1: 'Negative (-1)', 
                    0: 'Neutral (0)', 1: 'Positive (+1)', 2: 'Very Positive (+2)'}
colors = ['#d62728', '#ff7f0e', '#7f7f7f', '#2ca02c', '#1f77b4']

fig = go.Figure(data=[
    go.Bar(x=[sentiment_labels[i] for i in sentiment_counts.index],
           y=sentiment_counts.values,
           marker_color=colors,
           hovertemplate='%{x}<br>Count: %{y}')
])

fig.update_layout(
    title='Overall Sentiment Distribution',
    xaxis_title='Sentiment',
    yaxis_title='Count',
    height=350,
    showlegend=False
)
fig.show()

print(f"\nSentiment Summary:")
for s, count in sentiment_counts.items():
    pct = 100 * count / len(df_clean)
    print(f"  {sentiment_labels[s]}: {count} ({pct:.1f}%)")


Sentiment Summary:
  Very Negative (-2): 4 (0.1%)
  Negative (-1): 323 (6.7%)
  Neutral (0): 317 (6.6%)
  Positive (+1): 2451 (50.8%)
  Very Positive (+2): 1732 (35.9%)


In [5]:
# 5. Plot Average Sentiment by Calendar Month
# Aggregate by month (across all years)
month_agg = df_clean.groupby('month').agg({
    'qwen_sentiment': ['mean', 'count', 'std'],
    'post_id': 'count'
}).round(3)

month_agg.columns = ['avg_sentiment', 'sent_count', 'sentiment_std', 'total_posts']
month_agg['month_name'] = month_agg.index.map(month_names)
month_agg = month_agg.sort_index()

print("Monthly Sentiment Summary (averaged across all years):")
print(month_agg)

# Plot average sentiment by month
fig = go.Figure()

# Add line with markers for average sentiment
fig.add_trace(go.Scatter(
    x=month_agg['month_name'],
    y=month_agg['avg_sentiment'],
    mode='lines+markers',
    name='Avg Sentiment',
    line=dict(width=3, color='steelblue'),
    marker=dict(size=8),
    hovertemplate='%{x}<br>Avg Sentiment: %{y:.2f}<br>Posts: %{customdata}',
    customdata=month_agg['total_posts']
))

# Add confidence bands (±1 std)
fig.add_trace(go.Scatter(
    x=month_agg['month_name'],
    y=month_agg['avg_sentiment'] + month_agg['sentiment_std'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=month_agg['month_name'],
    y=month_agg['avg_sentiment'] - month_agg['sentiment_std'],
    mode='lines',
    line=dict(width=0),
    fillcolor='rgba(0,100,200,0.2)',
    fill='tonexty',
    name='±1 Std Dev',
    hoverinfo='skip'
))

fig.update_layout(
    title='Average Sentiment by Calendar Month (across all years)',
    xaxis_title='Month',
    yaxis_title='Average Sentiment Score',
    height=400,
    hovermode='x unified'
)
fig.show()

Monthly Sentiment Summary (averaged across all years):
       avg_sentiment  sent_count  sentiment_std  total_posts month_name
month                                                                  
1.0            1.311         505          0.802          505        Jan
2.0            1.241         311          0.809          311        Feb
3.0            0.984         256          0.872          256        Mar
4.0            1.076         396          0.742          396        Apr
5.0            1.178         477          0.658          477        May
6.0            1.026         273          0.838          273        Jun
7.0            0.983         346          0.897          346        Jul
8.0            1.007         298          0.928          298        Aug
9.0            0.976         288          0.931          288        Sep
10.0           0.953         298          0.994          298        Oct
11.0           1.191         346          0.857          346        Nov
12.0     

In [10]:
# 6. Filter and Analyze `-2` Sentiment Rows
df_neg2 = df_clean[df_clean['qwen_sentiment'] == -2].copy()
print(f"Found {len(df_neg2)} posts with -2 (very negative) sentiment")
print(f"Percentage of total: {100 * len(df_neg2) / len(df_clean):.1f}%")

# Monthly frequency for -2 sentiment
neg2_by_month = df_neg2.groupby('month').size().reindex(range(1, 13), fill_value=0)
neg2_by_month.index = neg2_by_month.index.map(month_names)

total_by_month = df_clean.groupby('month').size().reindex(range(1, 13), fill_value=0)
total_by_month.index = total_by_month.index.map(month_names)

# Calculate proportion
neg2_prop = (neg2_by_month / total_by_month * 100).round(1)

summary_df = pd.DataFrame({
    '-2 Count': neg2_by_month,
    'Total Posts': total_by_month,
    'Proportion %': neg2_prop
})

print("\n-2 Sentiment Distribution by Month:")
print(summary_df)
print(f"\nBucket distribution for -2 sentiment posts:")
print(df_neg2['qwen_bucket'].value_counts())

Found 4 posts with -2 (very negative) sentiment
Percentage of total: 0.1%

-2 Sentiment Distribution by Month:
       -2 Count  Total Posts  Proportion %
month                                     
Jan           0          505           0.0
Feb           0          311           0.0
Mar           1          256           0.4
Apr           0          396           0.0
May           0          477           0.0
Jun           2          273           0.7
Jul           0          346           0.0
Aug           0          298           0.0
Sep           0          288           0.0
Oct           0          298           0.0
Nov           0          346           0.0
Dec           1         1032           0.1

Bucket distribution for -2 sentiment posts:
qwen_bucket
批评讽刺    4
Name: count, dtype: int64


In [9]:
# 7. Bucket Distribution for `-2` Sentiment Posts
# Overall bucket distribution for -2 sentiment (no month breakdown)
neg2_buckets = df_neg2['qwen_bucket'].value_counts().sort_values(ascending=False)

print("Bucket distribution for -2 sentiment posts:")
print(neg2_buckets)
print(f"\nTotal -2 posts: {len(df_neg2)}")

# Bar chart
fig = go.Figure(data=[
    go.Bar(x=neg2_buckets.index,
           y=neg2_buckets.values,
           marker_color='#d62728',
           hovertemplate='%{x}<br>Count: %{y}')
])

fig.update_layout(
    title='Bucket Distribution for -2 Sentiment Posts',
    xaxis_title='Bucket',
    yaxis_title='Count',
    height=400,
    showlegend=False,
    xaxis_tickangle=-45
)
fig.show()

Bucket distribution for -2 sentiment posts:
qwen_bucket
批评讽刺    4
Name: count, dtype: int64

Total -2 posts: 4


In [ ]:
# 8. Export Figures and Aggregated Tables
import os
os.makedirs('results/summaries', exist_ok=True)

# Save monthly summary
month_agg.to_csv('results/summaries/monthly_sentiment_summary.csv')
print("✓ Saved: results/summaries/monthly_sentiment_summary.csv")

# Save -2 sentiment summary
summary_df.to_csv('results/summaries/neg2_sentiment_by_month.csv')
print("✓ Saved: results/summaries/neg2_sentiment_by_month.csv")

# Save -2 bucket crosstab
neg2_month_bucket.to_csv('results/summaries/neg2_bucket_month_crosstab.csv')
print("✓ Saved: results/summaries/neg2_bucket_month_crosstab.csv")

# Save sample of -2 posts
sample_neg2 = df_neg2[['post_id', 'time', 'text', 'qwen_bucket', 'qwen_confidence']].head(20)
sample_neg2.to_csv('results/summaries/neg2_sample_posts.csv', index=False)
print("✓ Saved: results/summaries/neg2_sample_posts.csv (first 20 posts)")

print(f"\n📊 Summary Statistics:")
print(f"   Total analyzed posts: {len(df_clean):,}")
print(f"   Very positive (+2): {(df_clean['qwen_sentiment'] == 2).sum()} ({100*(df_clean['qwen_sentiment'] == 2).sum()/len(df_clean):.1f}%)")
print(f"   Positive (+1): {(df_clean['qwen_sentiment'] == 1).sum()} ({100*(df_clean['qwen_sentiment'] == 1).sum()/len(df_clean):.1f}%)")
print(f"   Neutral (0): {(df_clean['qwen_sentiment'] == 0).sum()} ({100*(df_clean['qwen_sentiment'] == 0).sum()/len(df_clean):.1f}%)")
print(f"   Negative (-1): {(df_clean['qwen_sentiment'] == -1).sum()} ({100*(df_clean['qwen_sentiment'] == -1).sum()/len(df_clean):.1f}%)")
print(f"   Very negative (-2): {(df_clean['qwen_sentiment'] == -2).sum()} ({100*(df_clean['qwen_sentiment'] == -2).sum()/len(df_clean):.1f}%)")
print(f"\n📈 Monthly Pattern:")
print(f"   Peak sentiment month: {month_agg['avg_sentiment'].idxmax()} ({month_names[month_agg['avg_sentiment'].idxmax()]}) - {month_agg['avg_sentiment'].max():.2f}")
print(f"   Lowest sentiment month: {month_agg['avg_sentiment'].idxmin()} ({month_names[month_agg['avg_sentiment'].idxmin()]}) - {month_agg['avg_sentiment'].min():.2f}")